# Microduck RL — 롤러스케이트 세션 (Kaggle 무료 T4)

- 과제: **`Mjlab-Velocity-Flat-MicroDuck-Rollers`** (발 밑 수동 바퀴 4개, 평지). cmd_x: 0 = 서기/미끄러지기, + = 밀기, − = 브레이크.
- **MODE = "roller_warm"** (기본): Pollen 공개 `roller.onnx`(이미 0.3 m/s 정도로 탐)를 rsl_rl 체크포인트로 바꿔(`onnx_to_ckpt.py`, action_scale 0.8 을 마지막 층에 접어 넣음) **warm start** (`MICRODUCK_WARM_START=1`: 반복 수·커리큘럼은 0부터). actor 만 옮겨지고 critic 은 새로 배움.
  - 박스 CPU 시험(64 env, 4회): warm 평균 보상 +0.09 → +1.31, wheel_speed 0 → 0.066 / 처음부터는 −12.6 → −18.3, wheel_speed 0.004.
- **MODE = "roller_scratch"**: 처음부터 (std 1.0). 쓸 만한 타기까지 3–5천 회 = 대략 8–12 GPU 시간 예상.
- 목표: 공개 roller(시뮬 측정: cmd 0.4 → 0.30 m/s, 0.6 → 0.33 m/s)보다 **빠르고 명령에 맞게** 밀기 + 직진 유지(학습 보상 heading_hold).
- wandb 없음, Hugging Face 계정·업로드 없음(공개 파일 읽기만), 비용 0원. **30분마다 `snapshot_model_N.pt`**, **10시간에 스스로 멈춤**, 50회마다 체크포인트, 끝에 ONNX + 속도 측정.


In [ ]:
# ==== 설정 ====
MODE = "roller_warm"            # "roller_warm" | "roller_scratch"
NUM_ENVS = 2048                 # 메모리 부족이면 1024 → 512 로 자동 재시도
MAX_ITERATIONS = 6000           # 시간 예산이 먼저 끊음 (T4 2048env 약 6–9초/회 예상 → 10시간 ≈ 4–6천 회)
SAVE_INTERVAL = 50
TIME_BUDGET_H = 10.0            # 10시간 자동 정지 (주 30시간 무료 중). 춤 학습과 나눠 쓰려면 5.0 정도로
SNAPSHOT_EVERY_MIN = 30         # 학습 중에도 최신 체크포인트를 출력 폴더에 복사 (세션이 끊겨도 남도록)
WARM_STD = 0.3                  # warm start 행동 잡음 (처음부터는 1.0)
RESUME_FROM = None              # 이전 세션 출력 .pt 를 데이터셋으로 붙였으면 그 경로 (이 경우 변환 생략)
REPO = "https://github.com/pollen-robotics/microduck_rl.git"
BRANCH = "develop"
COMMIT = "bae39f5"              # 롤러 작업은 세션2 패치와 무관. 박스에서 변환·warm start 시험한 커밋


In [ ]:
import os, pathlib, subprocess, sys
OUT = pathlib.Path("/kaggle/working/out") if os.path.isdir("/kaggle/working") else pathlib.Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
WORK = pathlib.Path("/kaggle/temp" if os.path.isdir("/kaggle") else "/content") / "work"
WORK.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT, "| work dir:", WORK)
!nvidia-smi

In [ ]:
!pip -q install uv
!cd {WORK} && (test -d microduck_rl || git clone -b {BRANCH} {REPO} microduck_rl)
if COMMIT:
    !cd {WORK}/microduck_rl && git checkout -q {COMMIT}
!cd {WORK}/microduck_rl && git log --oneline -1 && uv sync 2>&1 | tail -3
!cd {WORK}/microduck_rl && uv run --no-sync python -c "import torch,warp as wp; print('torch', torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'); wp.init(); print('warp devices', wp.get_devices())"


In [ ]:
# 도구 파일 쓰기 (이 노트북 폴더의 onnx_to_ckpt.py / roller_probe.py 와 같은 내용)
O2C = r'''"""Pollen 공개 roller.onnx → rsl_rl 체크포인트(.pt) 변환 (warm start 용, 무료·CPU 몇 초).

roller.onnx = Sub(mean) → Div(std) → MLP 61→512→256→128→14 (ELU). 이 저장소 롤러 작업의 actor 와 같은 모양.
  - actor 가중치·관측 정규화(mean/std)를 ONNX 에서 복사
  - manifest: roller 정책은 action_scale 0.8, 학습 환경(Mjlab-Velocity-Flat-MicroDuck-Rollers)은 1.0
    → 마지막 층(mlp.6) 가중치·편향 × 0.8 로 접어 넣어 '환경 행동 = 0.8 × ONNX 출력' 을 그대로 재현
  - 행동 잡음 std = --std (기본 0.3; 처음부터 학습은 1.0) — 이미 타는 정책을 흔들지 않게 작게
  - critic(가치 함수)은 템플릿(1회 학습한 새 모델) 그대로 = 처음부터 배움. optimizer 상태도 템플릿 것.
  - 정규화 count 를 크게(1e6) 해서 학습 중 mean/std 가 천천히만 움직이게
템플릿 = 같은 작업으로 `--agent.max-iterations 1` 돌려 나온 model_0.pt (구조·optimizer 모양을 맞추려고).

  python onnx_to_ckpt.py roller.onnx template_model_0.pt out_roller_init.pt [--check]
"""
import argparse
import numpy as np
import onnx
from onnx import numpy_helper
import torch

ap = argparse.ArgumentParser()
ap.add_argument("onnx_file"); ap.add_argument("template"); ap.add_argument("out")
ap.add_argument("--action-scale", type=float, default=0.8)
ap.add_argument("--std", type=float, default=0.3)
ap.add_argument("--check", action="store_true", help="onnxruntime 출력 × 0.8 과 torch actor 출력 비교")
a = ap.parse_args()

m = onnx.load(a.onnx_file)
W = {i.name: numpy_helper.to_array(i) for i in m.graph.initializer}
div = [n for n in m.graph.node if n.op_type == "Div"][0]
std_name = [x for x in div.input if x in W][0]
mean, std = W["obs_normalizer._mean"].astype(np.float32), W[std_name].astype(np.float32)

ck = torch.load(a.template, map_location="cpu", weights_only=False)
sd = ck["actor_state_dict"]
for k in ("mlp.0.weight", "mlp.0.bias", "mlp.2.weight", "mlp.2.bias", "mlp.4.weight", "mlp.4.bias", "mlp.6.weight", "mlp.6.bias"):
    v = torch.from_numpy(W[k].copy())
    assert sd[k].shape == v.shape, (k, sd[k].shape, v.shape)
    sd[k] = v * a.action_scale if k.startswith("mlp.6") else v
sd["obs_normalizer._mean"] = torch.from_numpy(mean).reshape(sd["obs_normalizer._mean"].shape)
sd["obs_normalizer._std"] = torch.from_numpy(std).reshape(sd["obs_normalizer._std"].shape)
sd["obs_normalizer._var"] = sd["obs_normalizer._std"] ** 2
sd["obs_normalizer.count"] = torch.tensor(1e6, dtype=sd["obs_normalizer.count"].dtype)
sd["distribution.std_param"] = torch.full_like(sd["distribution.std_param"], a.std)
ck["actor_state_dict"] = sd
ck["iter"] = 0
ck.setdefault("infos", {})["converted_from"] = f"{a.onnx_file} (action_scale {a.action_scale} folded into mlp.6)"
torch.save(ck, a.out)
print("saved", a.out)

if a.check:
    import onnxruntime as ort
    s = ort.InferenceSession(a.onnx_file, providers=["CPUExecutionProvider"])
    x = np.random.default_rng(0).normal(size=(256, 61)).astype(np.float32) * std + mean
    y_onnx = np.concatenate([s.run(None, {s.get_inputs()[0].name: x[i:i + 1]})[0] for i in range(len(x))]) * a.action_scale
    h = (torch.from_numpy(x) - sd["obs_normalizer._mean"]) / sd["obs_normalizer._std"]
    for i in (0, 2, 4):
        h = torch.nn.functional.elu(h @ sd[f"mlp.{i}.weight"].T + sd[f"mlp.{i}.bias"])
    y = (h @ sd["mlp.6.weight"].T + sd["mlp.6.bias"]).numpy()
    print("max |torch - 0.8*onnx| =", float(np.abs(y - y_onnx).max()), "(action range", float(np.abs(y_onnx).max()), ")")
'''
PROBE = r'''"""롤러 정책 속도·직진·자세 측정 (학습 없음). 새로 학습한 ONNX 와 Pollen 공개 roller.onnx 를 같은 조건으로 비교.

  python roller_probe.py NEW.onnx [--scale 1.0] [--ref roller.onnx --ref-scale 0.8] [--device cpu|cuda:0]
측정: 시작 방향 0 고정, 무작위화 끔, 1초 서 있기 → cmd_x(0.2/0.4/0.6) 8초 → 마지막 5초 평균 속도, 방향 흔들림, 넘어짐.
주의: 이 저장소에서 내보낸 ONNX 는 변환 때 0.8 을 마지막 층에 접어 넣었으므로 --scale 1.0 (Pollen 원본은 0.8).
"""
import argparse, math, json
import numpy as np, torch, onnxruntime as ort
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import load_env_cfg
import mjlab_microduck.tasks  # noqa

ap = argparse.ArgumentParser()
ap.add_argument("onnx"); ap.add_argument("--scale", type=float, default=1.0)
ap.add_argument("--ref"); ap.add_argument("--ref-scale", type=float, default=0.8)
ap.add_argument("--device", default="cpu"); ap.add_argument("--out")
a = ap.parse_args()

cfg = load_env_cfg("Mjlab-Velocity-Flat-MicroDuck-Rollers", play=True); cfg.scene.num_envs = 1
for k in [k for k in cfg.events if any(s in k for s in ("push", "prone", "topple", "randomize", "encoder_bias", "base_com"))]: cfg.events.pop(k)
for k in list(cfg.terminations):
    if k != "nan_state": cfg.terminations.pop(k)
cfg.episode_length_s = 1000.0
for k in list(cfg.curriculum): cfg.curriculum.pop(k)
for c in cfg.commands.values(): c.resampling_time_range = (1e9, 1e9); c.debug_vis = False
env = ManagerBasedRlEnv(cfg=cfg, device=a.device)
r = env.scene["robot"]; dt = env.step_dt


def run(sess, scale, cx, T=9.0):
    env.reset()
    st = torch.zeros(1, 13, device=env.device); st[0, :3] = r.data.root_link_pos_w[0]; st[0, 3] = 1.0
    r.write_root_state_to_sim(st)
    vx, yaw, gz_max = [], [], -1.0
    for k in range(int(T / dt)):
        o = env.observation_manager.compute()["actor"].cpu().numpy().astype(np.float32)
        o[:, 48:51] = (cx if k * dt > 1.0 else 0.0, 0.0, 0.0); o[:, 51:61] = 0.0
        act = sess.run(None, {sess.get_inputs()[0].name: o})[0]
        env.step(torch.from_numpy(act).to(env.device) * scale)
        q = r.data.root_link_quat_w[0].cpu().numpy()
        yaw.append(math.degrees(math.atan2(2 * (q[0] * q[3] + q[1] * q[2]), 1 - 2 * (q[2] ** 2 + q[3] ** 2))))
        vx.append(float(r.data.root_link_lin_vel_b[0, 0])); gz_max = max(gz_max, float(r.data.projected_gravity_b[0, 2]))
    n = int(4.0 / dt)
    return dict(cmd_x=cx, mean_vx_last5s=round(float(np.mean(vx[n:])), 3), max_abs_yaw_deg=round(float(np.max(np.abs(yaw[n:]))), 1),
                upright=bool(gz_max < -0.8), max_tilt_gz=round(gz_max, 2))


res = {}
for name, path, sc in (("new", a.onnx, a.scale), ("pollen_roller", a.ref, a.ref_scale)):
    if not path:
        continue
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    res[name] = [run(s, sc, cx) for cx in (0.2, 0.4, 0.6)]
    for x in res[name]: print(name, x, flush=True)
if a.out:
    json.dump(res, open(a.out, "w"), indent=1)
'''
(WORK / "microduck_rl/onnx_to_ckpt.py").write_text(O2C)
(WORK / "microduck_rl/roller_probe.py").write_text(PROBE)
print("ok")

In [ ]:
# warm start 준비: 공개 roller.onnx 받기(읽기만) → 1회 학습으로 템플릿 → 변환
import glob, shutil
TASK = "Mjlab-Velocity-Flat-MicroDuck-Rollers"
env = dict(os.environ, WANDB_MODE="disabled", CUDA_VISIBLE_DEVICES="0")
R = WORK / "microduck_rl"
ROLLER_ONNX = subprocess.run(["uv", "run", "--no-sync", "python", "-c",
    "from huggingface_hub import hf_hub_download; print(hf_hub_download('pollen-robotics/microduck-policies', 'roller.onnx'))"],
    cwd=R, env=env, capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
print("roller.onnx:", ROLLER_ONNX)
INIT = None
if RESUME_FROM:
    INIT = RESUME_FROM
elif MODE == "roller_warm":
    subprocess.run(["uv", "run", "--no-sync", "train", TASK, "--agent.logger", "tensorboard", "--env.scene.num-envs", "64",
                    "--agent.max-iterations", "1", "--agent.run-name", "template"], cwd=R, env=env, check=True,
                   stdout=subprocess.DEVNULL)
    tpl = sorted(glob.glob(str(R / "logs/rsl_rl/velocity_rollers/*template*/model_0.pt")))[-1]
    INIT = str(WORK / "roller_init.pt")
    subprocess.run(["uv", "run", "--no-sync", "python", "onnx_to_ckpt.py", ROLLER_ONNX, tpl, INIT, "--std", str(WARM_STD), "--check"],
                   cwd=R, env=env, check=True)
    for d in glob.glob(str(R / "logs/rsl_rl/velocity_rollers/*template*")):
        shutil.rmtree(d)   # 템플릿 실행 폴더는 지움 (스냅샷이 헷갈리지 않게)
print("init checkpoint:", INIT)

In [ ]:
import shlex, time, re
if INIT:
    env["MICRODUCK_WARM_START"] = "1"
def build_cmd(num_envs):
    return (["uv", "run", "--no-sync", "train", TASK, "--agent.logger", "tensorboard",
             "--env.scene.num-envs", str(num_envs), "--agent.max-iterations", str(MAX_ITERATIONS),
             "--agent.save-interval", str(SAVE_INTERVAL), "--agent.run-name", f"free_{MODE}"]
            + (["--init-checkpoint", INIT] if INIT else []))
def snapshot():
    pts = glob.glob(str(R / "logs/rsl_rl/velocity_rollers/*/model_*.pt"))
    if pts:
        last = max(pts, key=os.path.getmtime)
        for old in glob.glob(str(OUT / "snapshot_model_*.pt")): os.remove(old)
        shutil.copy(last, OUT / ("snapshot_" + os.path.basename(last)))
t0 = time.time(); t_snap = t0
for n_env in [NUM_ENVS, 1024, 512]:
    cmd = build_cmd(n_env)
    print(" ".join(shlex.quote(c) for c in cmd))
    log = open(OUT / "train.log", "a"); log.write(f"=== try num_envs={n_env}\n")
    p = subprocess.Popen(cmd, cwd=R, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            log.write(line); log.flush()
            if any(k in line for k in ("Learning iteration", "Mean reward", "wheel_speed", "Iteration time", "Error", "Traceback", "checkpoints]")):
                print(line.rstrip())
            if time.time() - t_snap > SNAPSHOT_EVERY_MIN * 60:
                snapshot(); t_snap = time.time()
            if time.time() - t0 > TIME_BUDGET_H * 3600:
                print("time budget reached, stopping"); p.terminate(); break
    finally:
        p.wait(); log.close()
    snapshot()
    print("exit", p.returncode, "elapsed h", round((time.time() - t0) / 3600, 2))
    oom = "Failed to allocate" in open(OUT / "train.log").read().split("=== try")[-1]
    if p.returncode == 0 or not oom: break
    print("OOM at", n_env, "envs -> retry smaller")

In [ ]:
import statistics
txt = open(OUT / 'train.log').read()
it = [float(x) for x in re.findall(r'Iteration time:\s*([\d.]+)s', txt)]
ws = [float(x) for x in re.findall(r'Episode_Reward/wheel_speed:\s*([-\d.]+)', txt)]
print('iterations logged', len(it))
if it: print('iter time median', statistics.median(it), 's')
if ws: print('wheel_speed first/last', ws[0], ws[-1])
print(txt[-3000:])

In [ ]:
# 최신 체크포인트 → ONNX (0.8 은 이미 접혀 있으므로 이 ONNX 는 action_scale 1.0 으로 씀) + 속도 측정
runs = sorted(glob.glob(str(R / "logs/rsl_rl/velocity_rollers/*")), key=os.path.getmtime)
run = runs[-1]; print("run dir:", run)
pts = sorted(glob.glob(run + "/model_*.pt"), key=lambda f: int(re.findall(r"model_(\d+)", f)[0]))
last = pts[-1]; print("last checkpoint:", last)
onnx = OUT / (pathlib.Path(run).name + "_" + pathlib.Path(last).stem + ".onnx")
subprocess.run(["uv", "run", "--no-sync", "python", "scripts/export.py", TASK, "--checkpoint-file", last, "--onnx-file", str(onnx)],
               cwd=R, env=env, check=True)
shutil.copy(last, OUT)
for f in glob.glob(run + "/events.out.tfevents*") + glob.glob(run + "/params/*"):
    shutil.copy(f, OUT)
subprocess.run(["uv", "run", "--no-sync", "python", "roller_probe.py", str(onnx), "--scale", "1.0", "--ref", ROLLER_ONNX,
                "--ref-scale", "0.8", "--out", str(OUT / "roller_probe.json")], cwd=R, env=env, check=False)
!ls -la {OUT}

결과는 Output 탭 `/kaggle/working/out` 에 남아요 (`model_N.pt`, `snapshot_model_N.pt`, ONNX, `roller_probe.json`, `train.log`). **Save Version → Save & Run All (Commit)** 으로 돌리면 브라우저를 닫아도 끝까지 돌아요. 다음 세션은 이 출력 `.pt` 를 비공개 데이터셋으로 붙이고 `RESUME_FROM` 에 경로를 넣으면 이어서 학습돼요. `.pt` 는 GitHub 에 올리지 않아요.